# Capítulo 11 — Transporte de Calor e Massa em Corpos Hídricos

**Autor:** Jader Lugon Junior  
**Livro:** Fenômenos de Transporte: Fundamentos e Modelagem Computacional  
**Repositório:** [github.com/JaderLugon/fenomenos-transporte-livro](https://github.com/JaderLugon/fenomenos-transporte-livro)

---

## 🎯 Objetivos de Aprendizagem

Ao final deste notebook, você será capaz de:

1. **Estabelecer** a analogia formal entre transporte de massa e energia
2. **Calcular** coeficientes de dispersão hidrodinâmica (Liu, Elder, Jobson-Sayre)
3. **Determinar** o número de Péclet e classificar regimes de transporte
4. **Estimar** comprimentos de mistura transversal
5. **Aplicar** soluções analíticas para lançamentos instantâneos e contínuos
6. **Implementar** esquemas numéricos para advecção-dispersão

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import erfc
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
print('Ambiente configurado com sucesso!')

---

## 📚 Conteúdo Teórico Resumido

### 11.1 Analogia Formal

As equações de transporte de massa e energia têm **estrutura matemática idêntica**:

| Grandeza | Equação 1D | Propriedade Conservada |
|----------|-----------|------------------------|
| Massa | $\frac{\partial C}{\partial t} + u\frac{\partial C}{\partial x} = E\frac{\partial^2 C}{\partial x^2}$ | Concentração $C$ [kg/m³] |
| Energia | $\frac{\partial T}{\partial t} + u\frac{\partial T}{\partial x} = \alpha\frac{\partial^2 T}{\partial x^2}$ | Temperatura $T$ [K] |

### 11.2 Dispersão Hidrodinâmica

**Longitudinal (Liu, 1977):**
$$E_L = 0,011 \frac{U^2 B^2}{D \cdot U_*}$$

**Transversal (Elder, 1959):**
$$E_T = 0,23 \cdot D \cdot U_*$$

**Vertical (Jobson-Sayre):**
$$E_V = 0,067 \cdot D \cdot U_*$$

### 11.3 Número de Péclet

$$\mathrm{Pe} = \frac{U \cdot L}{E}$$

- $\mathrm{Pe} \gg 1$: advecção dominante
- $\mathrm{Pe} \ll 1$: difusão dominante
- $\mathrm{Pe} \approx 1$: mecanismos comparáveis

### 11.4 Comprimento de Mistura Transversal

$$L_T = K \frac{U \cdot B^2}{E_T}$$

onde $K \approx 0,28$ (margem) ou $K \approx 0,10$ (centro).

### 11.5 Soluções Analíticas

**Lançamento instantâneo (1D):**
$$C(x,t) = \frac{M}{A\sqrt{4\pi E_L t}} \exp\left[-\frac{(x - Ut)^2}{4E_L t}\right]$$

**Lançamento contínuo (1D):**
$$C(x,t) = \frac{C_0}{2} \left\{ \mathrm{erfc}\left[\frac{x - Ut}{\sqrt{4E_L t}}\right] + \exp\left(\frac{Ux}{E_L}\right) \mathrm{erfc}\left[\frac{x + Ut}{\sqrt{4E_L t}}\right] \right\}$$

---

## 🌊 Seção 1: Coeficientes de Dispersão

### Exercício 1.1: Cálculo de $E_L$ e $E_T$

Para um rio com $U = 0,4$ m/s, $B = 30$ m, $D = 1,2$ m, $S_0 = 0,0003$, calcule os coeficientes de dispersão.

In [ ]:
U = 0.4
B = 30.0
D = 1.2
S0 = 0.0003
g = 9.81

U_star = np.sqrt(g * D * S0)
E_L = 0.011 * (U**2 * B**2) / (D * U_star)
E_T = 0.23 * D * U_star
E_V = 0.067 * D * U_star

print('Coeficientes de Dispersao:')
print('  U* = {:.4f} m/s'.format(U_star))
print('  E_L = {:.2f} m2/s (longitudinal)'.format(E_L))
print('  E_T = {:.4f} m2/s (transversal)'.format(E_T))
print('  E_V = {:.4f} m2/s (vertical)'.format(E_V))
print('  Razao E_L/E_T = {:.0f}'.format(E_L / E_T))

---

## 📊 Seção 2: Número de Péclet

### Exercício 2.1: Classificação de Regime

Calcule $\mathrm{Pe}$ para o Rio Macaé com $L = 100$ m e $E_L = 12,57$ m²/s.

In [ ]:
U_rio = 0.20
L_rio = 100.0
E_L_rio = 12.57

Pe = U_rio * L_rio / E_L_rio

print('Numero de Peclet:')
print('  Pe = {:.2f}'.format(Pe))
if Pe > 10:
    print('  Regime: ADVECCAO DOMINANTE')
elif Pe < 0.1:
    print('  Regime: DIFUSAO DOMINANTE')
else:
    print('  Regime: MISTO (adveccao e difusao comparaveis)')

---

## 📏 Seção 3: Comprimento de Mistura Transversal

### Exercício 3.1: Estimativa de $L_T$

Para lançamento na margem do Rio Macaé ($B = 42,2$ m, $E_T = 0,00431$ m²/s), estime $L_T$.

In [ ]:
B_rio = 42.2
E_T_rio = 0.00431
K_margem = 0.28

L_T = K_margem * U_rio * B_rio**2 / E_T_rio
t_mistura = L_T / U_rio

print('Comprimento de Mistura Transversal:')
print('  L_T = {:.0f} m ({:.1f} km)'.format(L_T, L_T/1000))
print('  Tempo de mistura: {:.0f} h'.format(t_mistura/3600))

---

## 🧪 Seção 4: Soluções Analíticas

### Exercício 4.1: Lançamento Instantâneo

Simule um lançamento instantâneo de $M = 20$ kg em um rio com $U = 0,5$ m/s, $E_L = 8$ m²/s, $A = 37,5$ m².

In [ ]:
M = 20.0
U_sol = 0.5
E_L_sol = 8.0
A_sol = 37.5
t_sol = 7200  # 2 h

x_pico = U_sol * t_sol
C_max = M / (A_sol * np.sqrt(4 * np.pi * E_L_sol * t_sol))

print('Lancamento Instantaneo (t = {} h):'.format(t_sol/3600))
print('  Posicao do pico: {:.0f} m'.format(x_pico))
print('  Concentracao maxima: {:.4f} kg/m3 = {:.2f} mg/L'.format(C_max, C_max*1e6))

x_range = np.linspace(0, 5000, 200)
C_range = M / (A_sol * np.sqrt(4 * np.pi * E_L_sol * t_sol)) * np.exp(-(x_range - U_sol * t_sol)**2 / (4 * E_L_sol * t_sol))

fig, ax = plt.subplots()
ax.plot(x_range/1000, C_range*1e6, 'b-', lw=2)
ax.set_xlabel('Distancia x [km]')
ax.set_ylabel('Concentracao C [mg/L]')
ax.set_title('Pluma de Poluente em t = 2 h', fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 🔧 Seção 5: Esquema Numérico

### Exercício 5.1: Discretização

$$\frac{C_i^{n+1} - C_i^n}{\Delta t} + u\frac{C_i^n - C_{i-1}^n}{\Delta x} = E_L\frac{C_{i+1}^n - 2C_i^n + C_{i-1}^n}{\Delta x^2}$$

In [ ]:
N = 100
dx = 50.0
dt = 30.0
Co = U_sol * dt / dx
Di = E_L_sol * dt / dx**2

print('Parametros Adimensionais:')
print('  Courant (Co) = {:.3f}'.format(Co))
print('  Difusivo (Di) = {:.4f}'.format(Di))
print('  Peclet de celula = {:.2f}'.format(Co/Di))
print('\nEstabilidade:')
if Co <= 1 and Di <= 0.5:
    print('  ✅ Esquema explicito ESTAVEL')
else:
    print('  ❌ Esquema explicito INSTAVEL')

---

## 🔬 Estudos de Caso

| Estudo de Caso | Descrição | Link |
|----------------|-----------|------|
| **Caso 11.1** | Dispersão de Poluentes no Rio Macaé | [🔗 Abrir](../casos/11_1_Rio_Macae_Dispersao_Poluentes.ipynb) |
| **Caso 11.2** | Difusão Artificial em Esquemas Upwind | [🔗 Abrir](../casos/11_2_Difusao_Artificial_Upwind.ipynb) |

---

## 📖 Referências

- LIU, H. (1977). Predicting dispersion in streams. *Journal of Environmental Engineering*, 103(6), 961-974.
- ELDER, J. W. (1959). The dispersion of marked fluid in turbulent shear flow. *Journal of Fluid Mechanics*, 5(4), 544-560.
- FISCHER, H. B. et al. (1979). *Mixing in Inland and Coastal Waters*. Academic Press.
- LUGON JR., J. et al. (2008). Assessment of dispersion mechanisms in rivers. *Inverse Problems in Science and Engineering*, 16(8), 967-979.

---

## 🔄 Navegação

- [📚 Capítulo Anterior: Aletas](10_Aletas_Superficies.ipynb)
- [🏠 Repositório Principal](../README.md)

In [ ]:
print('Capitulo 11 concluido com sucesso!')